# Linear Regression Implementation
The start of every machine learning project begins the same way:
- Data Inspection
- Data Visualization
- Data Regularization
- Model Implementation

Our data is the [California Housing Dataset](https://www.kaggle.com/datasets/camnugent/california-housing-prices) from Kaggle. It contains 20,640 block groups from the 1990 California census. Each block group has features like median income, house age, average rooms, population, and geographic coordinates. We will predict the **median house value** for each block group.

---

## Phase 1: Data Inspection
We will first load a dataframe and inspect the data to understand its structure, types, and any missing values.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn as sk
from sklearn.model_selection import train_test_split
from sklearn import linear_model

data_df = pd.read_csv("housing.csv")

print("Shape is: ", data_df.shape)
print("\nColumn types:")
print(data_df.dtypes)
print("\nMissing values:")
print(data_df.isnull().sum())
data_df.head(10)

We have 20,640 samples with 9 numerical features and 1 categorical feature (`ocean_proximity`). There are 207 missing values in `total_bedrooms` which we will need to handle. The target variable is `median_house_value`.

Let's look at the basic statistics to understand the scale and distribution of each feature.

In [ ]:
data_df.describe()

A few things stand out:
- **Median income** ranges from ~0.5 to 15 (in tens of thousands of dollars)
- **Median house value** ranges from \$14,999 to \$500,001 (the cap at 500,001 suggests truncation)
- **Total rooms** and **population** have very large ranges, indicating significant variance across block groups
- Feature scales differ enormously (e.g., median_income ~0-15 vs total_rooms ~2-39,000), so normalization will be essential

---

## Phase 2: Data Visualization
The dataset has 8 numerical features and 1 output. We cannot visualize 8-dimensional data, but we can inspect each feature against the target to identify correlations.

In [ ]:
numerical_features = data_df.select_dtypes(include=[np.number]).columns.drop('median_house_value')

fig, axs = plt.subplots(4, 2, figsize=(20, 24))
fig.suptitle("Feature vs Median House Value", fontsize=24)

for idx, feature in enumerate(numerical_features):
    row, col = idx // 2, idx % 2
    axs[row, col].scatter(data_df[feature], data_df['median_house_value'], alpha=0.1, s=5)
    axs[row, col].set_xlabel(feature, fontsize=14)
    axs[row, col].set_ylabel('median_house_value', fontsize=14)

plt.tight_layout()
plt.show()

Key observations from the scatter plots:
- **Median income** shows the strongest positive correlation with house value - this will likely be our most important feature
- **Longitude and latitude** reveal geographic clustering - houses near the coast (specific lat/long bands) tend to be more expensive
- **Housing median age** shows a slight positive trend but with high variance
- The horizontal line at $500,001 confirms the value cap in the dataset
- **Total rooms, bedrooms, population, and households** show weak individual correlations with lots of spread

Let's visualize the geographic distribution of house values, since California geography (coastal vs inland) likely plays a major role in pricing.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
scatter = ax.scatter(data_df['longitude'], data_df['latitude'],
                     c=data_df['median_house_value'], cmap='magma',
                     alpha=0.4, s=10)
plt.colorbar(scatter, label='Median House Value ($)')
ax.set_xlabel('Longitude', fontsize=14)
ax.set_ylabel('Latitude', fontsize=14)
ax.set_title('California Housing Prices by Location', fontsize=18)
plt.tight_layout()
plt.show()

This is effectively a map of California. The brightest (most expensive) areas correspond to the San Francisco Bay Area and coastal Los Angeles/San Diego. Inland areas are consistently cheaper. This confirms that geography is a strong predictor of house value.

Let's also look at the categorical feature `ocean_proximity`.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(18, 6))

data_df['ocean_proximity'].value_counts().plot(kind='bar', ax=axs[0], color='steelblue')
axs[0].set_title('Ocean Proximity Distribution', fontsize=16)
axs[0].set_ylabel('Count', fontsize=14)
axs[0].tick_params(axis='x', rotation=45)

data_df.boxplot(column='median_house_value', by='ocean_proximity', ax=axs[1])
axs[1].set_title('House Value by Ocean Proximity', fontsize=16)
axs[1].set_ylabel('Median House Value ($)', fontsize=14)
axs[1].set_xlabel('Ocean Proximity', fontsize=14)
plt.suptitle('')

plt.tight_layout()
plt.show()

The boxplot confirms that `ISLAND` properties are the most expensive (though rare - only 5 samples), while `INLAND` properties are consistently the cheapest. `<1H OCEAN` and `NEAR BAY` are in the middle-to-upper range. This categorical feature carries useful information.